In [1]:
!python --version

Python 3.9.10


In [2]:
import torch

torch.__version__

'2.5.1+cpu'

In [3]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
from tqdm import tqdm

c:\Users\khana\OneDrive\Desktop\GPU t est in Tensorflow\.venv\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.10). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [4]:
!pip install matplotlib numpy

In [6]:
IMAGE_SIZE = 64

DATASET_PATH = r"D:\Genrative_AI\GANs\img_align_celeba\img_align_celeba"


def load_images():

    images=[]

    for file in tqdm(os.listdir(DATASET_PATH)):

        path=os.path.join(DATASET_PATH,file)

        try:

            img=Image.open(path)

            img=img.resize((IMAGE_SIZE,IMAGE_SIZE))

            img=np.array(img)

            images.append(img)


        except:
            pass


    images=np.array(images)

    return images

In [8]:
images = load_images()

images = images.astype("float32")

images = (images - 127.5) / 127.5

print(images.shape)
print(images.dtype)

100%|██████████| 202599/202599 [05:18<00:00, 636.31it/s]


(202599, 64, 64, 3)
float32


In [9]:
def build_generator():

    model=tf.keras.Sequential()


    model.add(
        layers.Dense(
            8*8*256,
            input_shape=(100,)
        )
    )


    model.add(layers.Reshape((8,8,256)))


    model.add(
        layers.BatchNormalization()
    )


    model.add(
        layers.Conv2DTranspose(
            128,
            kernel_size=4,
            strides=2,
            padding="same"
        )
    )


    model.add(
        layers.LeakyReLU()
    )


    model.add(
        layers.Conv2DTranspose(
            64,
            4,
            strides=2,
            padding="same"
        )
    )


    model.add(
        layers.LeakyReLU()
    )


    model.add(
        layers.Conv2DTranspose(
            3,
            4,
            strides=2,
            padding="same",
            activation="tanh"
        )
    )


    return model

In [10]:
generator=build_generator()

generator.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 16384)             1654784   
                                                                 
 reshape (Reshape)           (None, 8, 8, 256)         0         
                                                                 
 batch_normalization (BatchN  (None, 8, 8, 256)        1024      
 ormalization)                                                   
                                                                 
 conv2d_transpose (Conv2DTra  (None, 16, 16, 128)      524416    
 nspose)                                                         
                                                                 
 leaky_re_lu (LeakyReLU)     (None, 16, 16, 128)       0         
                                                                 
 conv2d_transpose_1 (Conv2DT  (None, 32, 32, 64)       1

In [11]:
def build_discriminator():

    model=tf.keras.Sequential()


    model.add(
        layers.Conv2D(
            64,
            4,
            strides=2,
            padding="same",
            input_shape=(64,64,3)
        )
    )


    model.add(
        layers.LeakyReLU()
    )


    model.add(
        layers.Conv2D(
            128,
            4,
            strides=2,
            padding="same"
        )
    )


    model.add(
        layers.LeakyReLU()
    )


    model.add(
        layers.Flatten()
    )


    model.add(
        layers.Dense(1,activation="sigmoid")
    )


    return model

In [12]:
discriminator=build_discriminator()

discriminator.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 64)        3136      
                                                                 
 leaky_re_lu_2 (LeakyReLU)   (None, 32, 32, 64)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 16, 16, 128)       131200    
                                                                 
 leaky_re_lu_3 (LeakyReLU)   (None, 16, 16, 128)       0         
                                                                 
 flatten (Flatten)           (None, 32768)             0         
                                                                 
 dense_1 (Dense)             (None, 1)                 32769     
                                                                 
Total params: 167,105
Trainable params: 167,105
Non-tr

In [13]:
discriminator.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)


discriminator.trainable=False


gan=tf.keras.Sequential(
[
generator,
discriminator
]
)


gan.compile(
optimizer="adam",
loss="binary_crossentropy"
)

In [14]:
EPOCHS=5000
BATCH_SIZE=32

noise_dim=100

In [15]:
def generate_images(epoch):

    noise=np.random.normal(
        0,
        1,
        (16,100)
    )


    fake_images=generator.predict(noise)


    fake_images=(fake_images+1)/2


    plt.figure(figsize=(4,4))


    for i in range(16):

        plt.subplot(4,4,i+1)

        plt.imshow(fake_images[i])

        plt.axis("off")


    plt.savefig(
        f"generated_images/{epoch}.png"
    )

    plt.close()

In [17]:
import os

# Create folders
os.makedirs("generated_images", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [19]:
for epoch in range(EPOCHS):


    idx=np.random.randint(
        0,
        images.shape[0],
        BATCH_SIZE
    )


    real_images=images[idx]


    noise=np.random.normal(
        0,
        1,
        (BATCH_SIZE,100)
    )


    fake_images=generator.predict(noise)



    real_labels=np.ones(
        (BATCH_SIZE,1)
    )


    fake_labels=np.zeros(
        (BATCH_SIZE,1)
    )



    d_loss_real=discriminator.train_on_batch(
        real_images,
        real_labels
    )


    d_loss_fake=discriminator.train_on_batch(
        fake_images,
        fake_labels
    )


    d_loss=(d_loss_real+d_loss_fake)/2



    noise=np.random.normal(
        0,
        1,
        (BATCH_SIZE,100)
    )


    g_loss=gan.train_on_batch(
        noise,
        real_labels
    )



    if epoch % 100 == 0:
        generate_images(epoch)

1/1 [==============================] - 0s 21ms/step
